# 3_modelling_full — Modelagem completa com target em log

Objetivos principais:
- usar `log1p(saleprice)` como target;
- construir pipeline de pré-processamento (num + cat);
- treinar baseline (LightGBM se disponível) e avaliar no espaço log e original;
- aplicar correção de viés ao retransformar previsões;
- salvar pipeline e métricas.


In [1]:
# Imports
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# tenta LightGBM, senão RandomForest
try:
    import lightgbm as lgb
    Model = lgb.LGBMRegressor
    BACKEND = 'lightgbm'
except Exception:
    from sklearn.ensemble import RandomForestRegressor
    Model = RandomForestRegressor
    BACKEND = 'sklearn'

print('model backend:', BACKEND)

model backend: lightgbm


In [2]:
# Carrega dados pós-limpeza (gerados pelo notebook data_cleaning)
DATA = Path('..') / 'data' / 'processed' / 'ames_clean.parquet'
assert DATA.exists(), f'Arquivo não encontrado: {DATA}'
df = pd.read_parquet(DATA)
print('df.shape =', df.shape)
display(df.head(2))

df.shape = (2925, 80)


,ms_subclass,ms_zoning,lot_frontage,lot_area,street,alley,lot_shape,land_contour,utilities,lot_config,...,pool_area,pool_qc,fence,misc_feature,misc_val,mo_sold,yr_sold,sale_type,sale_condition,saleprice
0,20,RL,141.0,31770,Pave,None,IR1,Lvl,AllPub,Corner,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,None,Reg,Lvl,AllPub,Inside,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000


## 1 — Por que usar log no target?
- reduz assimetria da distribuição do preço;
- diminui influência de outliers muito caros;
- estabiliza variância dos resíduos (erros) entre observações;
- facilita interpretar diferenças como variações percentuais;
- ao final convertemos com `np.expm1` e aplicamos correção de viés se desejado.

In [3]:
# Define y (orig) e y_log
y = df['saleprice'].astype(float)
y_log = np.log1p(y)
print('y min/max:', y.min(), y.max())
print('y_log sample:', np.round(y_log.iloc[:5],2))

y min/max: 12789.0 625000.0
y_log sample: 0    12.28
1    11.56
2    12.06
3    12.40
4    12.15
Name: saleprice, dtype: float64


## 2 — Seleção automática de features (baseado em data_cleaning)
Selecionamos: todas colunas numéricas (exceto `saleprice`, `yr_sold`) e todas colunas categóricas.
Você pode ajustar manualmente a lista para incluir apenas variáveis relevantes do EDA (ex.: `overall_qual`, `gr_liv_area`).

In [ ]:
# normaliza colunas textuais
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype('string')

# seleciona colunas numéricas e categóricas (inclui 'string' explicitamente)
num_cols = df.select_dtypes(include=['number']).columns.drop(['saleprice','yr_sold'], errors='ignore').tolist()
cat_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()
print('num_cols:', len(num_cols), 'cat_cols:', len(cat_cols))
print('num sample:', num_cols[:6])
print('cat sample:', cat_cols[:6])

num_cols: 34 cat_cols: 44
num sample: ['lot_frontage', 'lot_area', 'overall_qual', 'overall_cond', 'year_built', 'year_remod_add']
cat sample: ['ms_subclass', 'ms_zoning', 'street', 'alley', 'lot_shape', 'land_contour']


## 3 — Pré-processamento (pipeline)
- Numéricos: median imputer + standard scaler
- Categóricos: imputar 'None' + OneHotEncoder(handle_unknown='ignore')
Usamos `ColumnTransformer` para manter o fluxo reproduzível.

In [22]:
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(missing_values=pd.NA, fill_value='None', strategy='constant')),('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('num', num_pipeline, num_cols),('cat', cat_pipeline, cat_cols)], remainder='drop')
print('preprocessor OK')

preprocessor OK


## 4 — Split treino/teste
Por padrão usamos um split aleatório; se quiser validação temporal, separe por `yr_sold` conforme o EDA.

In [16]:
X = df.drop(columns=['saleprice']).copy()[num_cols + cat_cols]
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
print('split feito — treino/teste:', X_train.shape[0], X_test.shape[0])

split feito — treino/teste: 2340 585


## 5 — Monta pipeline final e treina modelo baseline
Pipeline: (`preprocessor`, `model`). Usamos LightGBM quando disponível; caso contrário RandomForest.

In [29]:
pipe = Pipeline([('preprocessor', preprocessor), ('model', Model(random_state=42) if BACKEND=='sklearn' else Model(n_estimators=100, random_state=42))])
pipe.fit(X_train, y_train_log)
print('treinamento concluído')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003987 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3810
[LightGBM] [Info] Number of data points in the train set: 2340, number of used features: 231
[LightGBM] [Info] Start training from score 12.018024
treinamento concluído


## 6 — Avaliação: espaço log e espaço original (com correção de viés)
1) RMSE no espaço log — avalia diretamente o erro na transformação;
2) Retransformação sem correção (simples `expm1`) — mostra erro na unidade original;
3) Retransformação com correção de viés: `exp(pred + 0.5 * sigma_resid**2)`
   — onde `sigma_resid` é o desvio padrão dos resíduos no conjunto de treino (em log).

In [30]:
# previsões em log
pred_log = pipe.predict(X_test)
rmse_log = root_mean_squared_error(y_test_log, pred_log)
print('RMSE (log):', round(rmse_log,4))

# retransforma sem correção
pred_orig = np.expm1(pred_log)
y_test_orig = np.expm1(y_test_log)
rmse_orig = root_mean_squared_error(y_test_orig, pred_orig)
print('RMSE (orig, sem correção):', round(rmse_orig,2))

# estima sigma dos resíduos no treino (log space)
train_pred_log = pipe.predict(X_train)
resid_train = y_train_log - train_pred_log
sigma_resid = resid_train.std(ddof=0)
print('sigma_resid (train, log):', round(sigma_resid,4))

# retransformação com correção de viés (log-normal)
pred_orig_bc = np.expm1(pred_log + 0.5 * sigma_resid**2)
rmse_orig_bc = root_mean_squared_error(y_test_orig, pred_orig_bc)
print('RMSE (orig, bias-corr):', round(rmse_orig_bc,2))

# métricas adicionais
r2_log = r2_score(y_test_log, pred_log)
r2_orig = r2_score(y_test_orig, pred_orig_bc)
print('R2 (log):', round(r2_log,3), 'R2 (orig, bias-corr):', round(r2_orig,3))

RMSE (log): 0.0935
RMSE (orig, sem correção): 18396.11
sigma_resid (train, log): 0.0577
RMSE (orig, bias-corr): 18345.22
R2 (log): 0.949 R2 (orig, bias-corr): 0.953


## 7 — Salvando pipeline e métricas
Salvamos o pipeline completo (`preprocessor` + `model`) e um arquivo `metrics.json` com as métricas principais.

In [31]:
OUT = Path('..') / 'models'
OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, OUT / 'pipeline_log.joblib')
metrics = {
  'rmse_log': float(rmse_log),
  'rmse_orig_no_corr': float(rmse_orig),
  'rmse_orig_biascorr': float(rmse_orig_bc),
  'sigma_resid_log': float(sigma_resid)
}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print('pipeline salvo em', OUT / 'pipeline_log.joblib')
print('metrics salvo em', OUT / 'metrics.json')

pipeline salvo em ../models/pipeline_log.joblib
metrics salvo em ../models/metrics.json
